# Solving Sudoku Puzzles

This project implements an automated Sudoku solver using Backtracking Search and Constraint Satisfaction. The goal is to efficiently solve 9x9 grids of varying difficulty, from "Very Easy" to "Hard."

## Project Overview

A Sudoku puzzle is solved when every empty cell (designated as 0) is filled with a number from 1 to 9 such that:

* Each row contains each digit exactly once.
* Each column contains each digit exactly once.
* Each 3x3 sub-grid contains each digit exactly once.

## The Algorithm

The current implementation utilizes a depth-first backtracking algorithm.

1. Validation: Before solving, the agent checks if the initial board state is valid.
2. Recursive Search: The solver iterates through empty cells, attempting to place a digit and recursively checking if that lead to a solution.
3. Backtracking: If a digit placement leads to an impossible state, the algorithm "backtracks," resets the cell, and tries the next possible digit.

<img src="images/sudoku.png" style="width: 50%;"/>

In [1]:
import numpy as np

# Load sudokus
sudoku = np.load("data/very_easy_puzzle.npy")
print("very_easy_puzzle.npy has been loaded into the variable sudoku")
print(f"sudoku.shape: {sudoku.shape}, sudoku[0].shape: {sudoku[0].shape}, sudoku.dtype: {sudoku.dtype}")

# Load solutions for demonstration
solutions = np.load("data/very_easy_solution.npy")
print()

# Print the first 9x9 sudoku...
print("First sudoku:")
print(sudoku[0], "\n")

# ...and its solution
print("Solution of first sudoku:")
print(solutions[0])

very_easy_puzzle.npy has been loaded into the variable sudoku
sudoku.shape: (15, 9, 9), sudoku[0].shape: (9, 9), sudoku.dtype: int8

First sudoku:
[[1 0 4 3 8 2 9 5 6]
 [2 0 5 4 6 7 1 3 8]
 [3 8 6 9 5 1 4 0 2]
 [4 6 1 5 2 3 8 9 7]
 [7 3 8 1 4 9 6 2 5]
 [9 5 2 8 7 6 3 1 4]
 [5 2 9 6 3 4 7 8 1]
 [6 0 7 2 9 8 5 4 3]
 [8 4 3 0 1 5 2 6 9]] 

Solution of first sudoku:
[[1 7 4 3 8 2 9 5 6]
 [2 9 5 4 6 7 1 3 8]
 [3 8 6 9 5 1 4 7 2]
 [4 6 1 5 2 3 8 9 7]
 [7 3 8 1 4 9 6 2 5]
 [9 5 2 8 7 6 3 1 4]
 [5 2 9 6 3 4 7 8 1]
 [6 1 7 2 9 8 5 4 3]
 [8 4 3 7 1 5 2 6 9]]


## Solver Implementation

In [ ]:
import numpy as np

def sudoku_solver(sudoku):
    """
    Solves a Sudoku puzzle using backtracking with the MRV heuristic.
    
    Returns:
        9x9 numpy array: The solution, or all -1s if unsolvable.
    """
    # Create a copy to avoid modifying the original input array
    solved_sudoku = sudoku.copy()
    
    # Check if the initial state is already invalid
    if not is_valid_initial_state(solved_sudoku):
        return np.full((9, 9), -1)
    
    if solve(solved_sudoku):
        return solved_sudoku
    else:
        return np.full((9, 9), -1)

def get_possibilities(sudoku, row, col):
    """Returns a set of allowed numbers for a specific cell."""
    if sudoku[row, col] != 0:
        return set()
    
    # Numbers 1-9
    allowed = set(range(1, 10))
    
    # Remove numbers present in the same row or column
    allowed -= set(sudoku[row, :])
    allowed -= set(sudoku[:, col])
    
    # Remove numbers present in the 3x3 square
    start_r, start_c = (row // 3) * 3, (col // 3) * 3
    square = sudoku[start_r:start_r+3, start_c:start_c+3]
    allowed -= set(square.flatten())
    
    return allowed

def find_mrv_cell(sudoku):
    """Finds the empty cell with the fewest remaining valid possibilities (MRV)."""
    best_cell = None
    min_possibilities = 10 
    
    for r in range(9):
        for c in range(9):
            if sudoku[r, c] == 0:
                possibilities = get_possibilities(sudoku, r, c)
                num_p = len(possibilities)
                
                # If a cell has 0 possibilities, this branch is a dead end
                if num_p == 0:
                    return (r, c), []
                
                if num_p < min_possibilities:
                    min_possibilities = num_p
                    best_cell = (r, c)
                    best_choices = list(possibilities)
                    
                # Optimization: if we find a cell with only 1 choice, take it immediately
                if min_possibilities == 1:
                    return best_cell, best_choices
                    
    return best_cell, (best_choices if best_cell else [])

def solve(sudoku):
    """Recursive backtracking function using MRV."""
    cell, choices = find_mrv_cell(sudoku)
    
    # If no empty cells are left, the puzzle is solved
    if cell is None:
        return True
    
    # If an empty cell has no choices, backtrack
    if not choices:
        return False
        
    row, col = cell
    for value in choices:
        sudoku[row, col] = value
        if solve(sudoku):
            return True
        # Backtrack: reset cell if the choice didn't lead to a solution
        sudoku[row, col] = 0
            
    return False

def is_valid_initial_state(sudoku):
    """Checks the 9x9 grid for any immediate rule violations."""
    for i in range(9):
        # Check rows and columns for duplicates (ignoring zeros)
        row = sudoku[i, sudoku[i, :] != 0]
        col = sudoku[sudoku[:, i] != 0, i]
        if len(row) != len(set(row)) or len(col) != len(set(col)):
            return False
            
    # Check all 3x3 blocks
    for r in range(0, 9, 3):
        for c in range(0, 9, 3):
            block = sudoku[r:r+3, c:c+3].flatten()
            block = block[block != 0]
            if len(block) != len(set(block)):
                return False
    return True



## Performance and Testing

To evaluate the efficiency of the solver, we test it against datasets of varying complexity. The solver is designed to handle:
* Valid Puzzles: Returning a fully populated 9x9 NumPy array.
* Invalid Puzzles: Returning a 9x9 array of -1 if the initial state is unsolvable.

### Example Usage

In [ ]:
SKIP_TESTS = False

if not SKIP_TESTS:
    import time
    import numpy as np
    __SCORES = {}   
    difficulties = ['very_easy', 'easy', 'medium', 'hard']

    for difficulty in difficulties:
        print(f"Testing {difficulty} sudokus")
        
        sudokus = np.load(f"data/{difficulty}_puzzle.npy")
        solutions = np.load(f"data/{difficulty}_solution.npy")
        
        count = 0
        for i in range(len(sudokus)):
            sudoku = sudokus[i].copy()
            print(f"This is {difficulty} sudoku number", i)
            print(sudoku)
            
            start_time = time.process_time()
            your_solution = sudoku_solver(sudoku)
            end_time = time.process_time()
            
            if not isinstance(your_solution, np.ndarray):
                print("\033[91m[ERROR] Your sudoku_solver function returned a variable that has the incorrect type. If you submit this it will likely fail the auto-marking procedure result in a mark of 0 as it is expecting the function to return a numpy array with a shape (9,9).\n\t\033[94mYour function returns a {} object when {} was expected.\n\x1b[m".format(type(your_solution), np.ndarray))
            elif not np.all(your_solution.shape == (9, 9)):
                print("\033[91m[ERROR] Your sudoku_solver function returned an array that has the incorrect shape.  If you submit this it will likely fail the auto-marking procedure result in a mark of 0 as it is expecting the function to return a numpy array with a shape (9,9).\n\t\033[94mYour function returns an array with shape {} when {} was expected.\n\x1b[m".format(your_solution.shape, (9, 9)))
            
            print(f"This is your solution for {difficulty} sudoku number", i)
            print(your_solution)
            
            print("Is your solution correct?")
            if np.array_equal(your_solution, solutions[i]):
                print("Yes! Correct solution.")
                count += 1
            else:
                print("No, the correct solution is:")
                print(solutions[i])
            
            print("This sudoku took {} seconds to solve.\n".format(end_time-start_time))

        print(f"{count}/{len(sudokus)} {difficulty} sudokus correct")
        __SCORES[difficulty] = {
            'correct': count,
            'total': len(sudokus)
        }

In [4]:
# This is a TEST CELL. Do not delete or change.